# Notebook 5 — Three-Class XLM-RoBERTa: Positive / Negative / Neutral

## Purpose
The binary XLM-RoBERTa model (Notebook 3) achieves 82% accuracy on the RUSAD dataset but
cannot express a third class for comments that carry no sentiment — neutral questions,
factual statements, and sarcastic/mixed examples that are ambiguous under a two-label scheme.
Low-confidence analysis of 4,138 collected YouTube comments found that ~10% fall below 70%
confidence, with four recurring failure patterns all rooted in the missing neutral class.

This notebook extends the model to **three classes (Positive / Negative / Neutral)** by:

1. Using the original RUSAD dataset (10,999 rows) for Positive and Negative examples
2. Pulling human-labelled Neutral examples from `annotations.csv` (the output of the annotation tool)
3. Setting aside Sarcastic/Mixed comments into `sarcasm_set.csv` for a separate future study
4. Handling the class imbalance (Neutral will be much smaller) using weighted cross-entropy loss
5. Fine-tuning `xlm-roberta-base` for sequence classification with `num_labels=3`

## Hyperparameters (same as Notebook 3)
| Parameter | Value |
|-----------|-------|
| Learning rate | 1e-5 |
| Batch size | 16 |
| Epochs | 5 |
| Warmup steps | 50 |
| Max sequence length | 128 |

## Label Encoding
| Class | Integer |
|-------|---------|
| Negative | 0 |
| Positive | 1 |
| Neutral | 2 |

## Data Sources
- `rusad_cleaned.csv` — preprocessed RUSAD dataset (label_encoded: 0=Negative, 1=Positive)
- `annotations.csv` — human labels from the annotation tool (Positive/Negative/Neutral/Sarcastic/Mixed)

## Output
- `xlmroberta_3class/` — saved model and tokenizer
- `xlmroberta_3class.zip` — zipped for download
- `ThreeClass_confusion_matrix.png` — per-class confusion matrix
- `sarcasm_set.csv` — held-out Sarcastic/Mixed comments for future study

In [ ]:
# Install dependencies (Colab pre-installs torch; add what may be missing)
!pip install -q transformers sentencepiece scikit-learn seaborn

In [ ]:
import os
import random
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Libraries loaded.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/roman_urdu_sentiment')
print('Working directory:', os.getcwd())

In [ ]:
# Load RUSAD — Positive (1) and Negative (0) examples
df_rusad = pd.read_csv('rusad_cleaned.csv')[['cleaned_text', 'label_encoded']].copy()
df_rusad.columns = ['text', 'label']
df_rusad['text'] = df_rusad['text'].fillna('').astype(str).str.strip()
df_rusad = df_rusad[df_rusad['text'] != ''].reset_index(drop=True)

print(f'RUSAD rows: {len(df_rusad)}')
print(df_rusad['label'].value_counts().rename({0: 'Negative', 1: 'Positive'}))

In [ ]:
# Load human annotations and split by label
df_ann = pd.read_csv('annotations.csv')
print('Human label distribution:')
print(df_ann['human_label'].value_counts())
print()

# Hold out Sarcastic/Mixed for a separate future study
df_sarcasm = df_ann[df_ann['human_label'] == 'Sarcastic/Mixed'][['text', 'model_prediction', 'model_confidence']].copy()
df_sarcasm.to_csv('sarcasm_set.csv', index=False)
print(f'Sarcastic/Mixed held out → sarcasm_set.csv ({len(df_sarcasm)} rows)')

# Extract Neutral examples — these become class 2
df_neutral = df_ann[df_ann['human_label'] == 'Neutral'][['text']].copy()
df_neutral['label'] = 2
df_neutral['text'] = df_neutral['text'].fillna('').astype(str).str.strip()
df_neutral = df_neutral[df_neutral['text'] != ''].reset_index(drop=True)
print(f'Neutral examples for training: {len(df_neutral)}')

if len(df_neutral) < 10:
    raise ValueError(
        f'Only {len(df_neutral)} Neutral examples found. '
        'Label more comments as Neutral in annotate.py before training.'
    )

In [ ]:
# Combine RUSAD (labels 0/1) and Neutral annotations (label 2)
df_combined = pd.concat([df_rusad, df_neutral], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=SEED).reset_index(drop=True)

label_names = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
print('Three-class dataset:')
for lbl, name in label_names.items():
    count = (df_combined['label'] == lbl).sum()
    print(f'  {name} (label={lbl}): {count:,} ({count / len(df_combined) * 100:.1f}%)')
print(f'  Total: {len(df_combined):,}')

In [ ]:
# Stratified train/test split (stratify ensures Neutral examples appear in both sets)
X = df_combined['text'].values
y = df_combined['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
for lbl, name in label_names.items():
    print(f'  Train {name}: {(y_train == lbl).sum():,}  |  Test {name}: {(y_test == lbl).sum():,}')

In [ ]:
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }


BATCH_SIZE   = 16
train_ds     = SentimentDataset(X_train, y_train)
test_ds      = SentimentDataset(X_test,  y_test)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load model with 3 output classes and matching label mapping
id2label = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)
model = model.to(device)
print('Model loaded.')

# Class weights — Neutral will have a much higher weight to compensate for imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=y_train,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print('Class weights:')
for lbl, name in id2label.items():
    print(f'  {name}: {class_weights[lbl]:.3f}')

In [ ]:
LR          = 1e-5
NUM_EPOCHS  = 5
WARMUP_STEPS = 50

optimizer         = AdamW(model.parameters(), lr=LR)
num_training_steps = NUM_EPOCHS * len(train_loader)
lr_scheduler      = get_scheduler(
    'linear',
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=num_training_steps,
)

print(f'Training for {NUM_EPOCHS} epochs ({num_training_steps} steps, {WARMUP_STEPS} warmup)\n')

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{NUM_EPOCHS}'):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        # Forward pass — do NOT pass labels to model so we control the loss
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} — avg loss: {avg_loss:.4f}')

print('\nTraining complete.')

In [ ]:
model.eval()
all_preds, all_labels_list = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Evaluating'):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels_list.extend(labels.cpu().numpy())

target_names = ['Negative', 'Positive', 'Neutral']
print(classification_report(all_labels_list, all_preds, target_names=target_names))

In [ ]:
cm = confusion_matrix(all_labels_list, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Purples',
    xticklabels=['Negative', 'Positive', 'Neutral'],
    yticklabels=['Negative', 'Positive', 'Neutral'],
    ax=ax,
)
ax.set_title('Three-Class XLM-RoBERTa — Confusion Matrix', pad=14)
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('ThreeClass_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → ThreeClass_confusion_matrix.png')

In [ ]:
SAVE_DIR = 'xlmroberta_3class'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Model and tokenizer saved to {SAVE_DIR}/')

In [ ]:
from google.colab import files

ZIP_PATH = 'xlmroberta_3class.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(SAVE_DIR):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            zf.write(filepath)

print(f'Zipped → {ZIP_PATH}')
files.download(ZIP_PATH)